# Table 2: the winning expressions, scored on every halo

Every row of Table 2 is a row of `global_fit/family_fits.csv`, whose `equation` is $\log_{10} f$ in Halo 2's
units. A halo's prediction in its own units is that minus its `folds.json` offset, which is
$C/(G^{3/2} r_s^3 \rho_s^{1/2})$ with the halo's own $(r_s, \rho_s)$. Each is scored against each halo's
raw DF on the CV grid; the mean over the 30 halos is the Av. Halo MSLE column.

Writes `output/table2_per_halo.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # AurigaDFs/, so the package imports
from Empirical_DF_Construction.io.results_paths import cv_dir
from Empirical_DF_Construction.paper_figures.common import load_cv, out_dir

FITS = pd.read_csv(cv_dir() / "global_fit" / "family_fits.csv").set_index("family_form")
ROWS = {"eps^ln10": "log(eps_hat)",                                       # Table 2, in order
        "eps^alpha": "C*log(eps_hat)",
        "Rank 1": "C + log(eps_hat) - log(1 - eps_hat)",
        "Rank 2": "C + C/eps_hat + log(eps_hat) - log(1 - eps_hat)"}
EPS = sp.Symbol("eps_hat")
LOG10F = {k: sp.lambdify(EPS, sp.sympify(FITS.loc[f, "equation"], locals={"eps_hat": EPS})) for k, f in ROWS.items()}
grid, raw, offsets = load_cv()

In [ ]:
per = pd.DataFrame([dict(halo=h, label=k, mse=float(np.mean((raw[h] - LOG10F[k](grid) + offsets[h]) ** 2)))
                    for h in raw for k in ROWS])
per.to_csv(out_dir() / "table2_per_halo.csv", index=False)
print("Av. Halo MSLE:", per.groupby("label", sort=False).mse.mean().round(4).to_dict())